---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [3]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[spotmediaro] Votăm Elena Lasconi Președinte speranța României 🇷🇴🇲🇩🇪🇺💪🏻💯💝🍀🙏🏻
[CălinGeorgescu-CanalulOficial] Sperăm ca poporul român să conștientizeze cât mai repede direcția în care se înd
[turcescu111] Robert Turcescu, felicitari pt emisiune. Pentru urmatoarele emisiuni, i-a în con


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte
 
AXA_1 = "geopolitical_frame"
AXA_2 = "national_identity_frame"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
AXA_1_DEFINITION = """
geopolitic măsoară dacă textul promovează o viziune izolaționistă sau anti-occidentală în detrimentul integrării europene și euro-atlantice.
0 = absent (textul este neutru sau susține cooperarea globală/europeană)
1 = prezent (textul critică moderat parteneriatele internaționale)
2 = dominant (textul promovează agresiv suveranismul radical și respinge explicit valorile occidentale)
"""
 
AXA_2_DEFINITION = """
national_identity măsoară dacă textul folosește argumente naționaliste rigide, etnocentrice sau xenofobe pentru a respinge diversitatea și cooperarea externă.
0 = absent (textul nu folosește identitatea națională ca pe o barieră împotriva altor culturi)
1 = prezent (textul folosește o retorică de protejare a identității naționale în fața influențelor externe)
2 = dominant (textul prezintă identitatea națională ca fiind pură și superioară, respingând complet multiculturalismul sau globalizarea)
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [7]:
MINI_PROMPT = f"""
Ești un expert în științe politice și analist media, specializat în analiza discursului politic, populismului și curentelor suveraniste/izolaționiste din mediul online.
 
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
 
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
 
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
 
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
 
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
 
print(MINI_PROMPT)


Ești un expert în științe politice și analist media, specializat în analiza discursului politic, populismului și curentelor suveraniste/izolaționiste din mediul online.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. geopolitical_frame
2. national_identity_frame

CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
geopolitical_frame = 0 / 1 / 2
national_identity_frame = 0 / 1 / 2

DEFINIȚII:

geopolitic măsoară dacă textul promovează o viziune izolaționistă sau anti-occidentală în detrimentul integrării europene și euro-atlantice.
0 = absent (textul este neutru sau susține cooperarea globală/europeană)
1 = prezent (textul critică moderat parteneriatele internaționale)
2 = dominant (textul promovează agresiv suveranismul radical și respinge explicit valorile occidentale)


national_identity măsoară dacă text

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [9]:
TESTS = corpus.sample(5, random_state=36)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
153,yt_spwpI2Dm_yc_Ugxv3B53EUSE_OeY-TZ4AaABAg,StirileProTV,Știrile PRO TV - 1 Aprilie 2026 | Medic aneste...,După ce Americani iau mîncat de bani pe țările...
48,yt_HDsCdSOtxO0_UgyVQx1-XMIhVEvxtAt4AaABAg,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii ...,Cel mai cinstit canal Record Bordel (recorder)...
262,yt_LaJHhVE344g_Ugz3d33EzOBgE9onrJ94AaABAg,RadioClasic,Lentila de contact cu Stelian Tănase - Radiogr...,Pai cum sa mai candideze? Nu e deja la al 2-le...
17,yt__PIes5dU7Zg_UgydLu5WTPhD3lnUl0h4AaABAg,turcescu111,Swingeri politici în acțiune,Puterea si opozitia sunt in mana aceluiasi reg...
412,yt_-IoCNoMYLbI_UgxH6Syhn2r86bPQAWF4AaABAg,spotmediaro,Corneliu Bjola: România a pierdut războiul hib...,Domnul Bjola este de-asemenea preferatul meu. ...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [10]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [11]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [12]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
După ce Americani iau mîncat de bani pe țările NATO și iau obligat să le cumpere armament ,acum face pe ,supăratul

OUTPUT MODEL:
```json
{
  "target": "Statele Unite ale Americii",
  "stance": "anti",
  "tone": "acuzator",
  "geopolitical_frame": 1,
  "national_identity_frame": 0
}
```
COMENTARIU:
Cel mai cinstit canal Record Bordel (recorder) strange bani pentru ,,ajutor pentru Ucraina" 😂

OUTPUT MODEL:
```json
{
  "target": "Recorder",
  "stance": "anti",
  "tone": "ironic",
  "geopolitical_frame": 1,
  "national_identity_frame": 0
}
```
COMENTARIU:
Pai cum sa mai candideze? Nu e deja la al 2-lea mandat? Apoi, are varsta pe care o avea Biden de radea (si) el. Daca in Consitutia SUA exista o limita minima pt candidatii la presedintie (35 ani) nu e normal si de bun simt sa fie si o limita maxima?!? Ca altfel uite cu ce te alegi 🎃 daca n-ai din ce alege... (ca la noi, din pacate!)

OUTPUT MODEL:
```json
{
  "target": "Joe Biden",
  "stance": "anti",
  "tone": "ironic",
  "g

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

Am ales axele geopolitic și national_identity pentru a cuantifica retorica izolaționistă și etnocentrică din comentarii. Modelul a returnat în final un format JSON corect și valid. Cea mai mare problemă a fost tehnică (erorile de Rate Limit și cheie API pe clientul standard), iar în prompt aș schimba granularitatea definițiilor pentru a diferenția mai clar scorurile 1 și 2.

### Mini-tipologia discursului (bazată pe axele alese)

Prin combinarea axelor `geopolitical_frame` (axa 1) și `national_identity_frame` (axa 2) pe o scală de la 0 la 2, putem defini 4 categorii clare de discurs identificate în comentarii:

1. **Integraționistul / Globalistul (Scor 0 pe ambele axe)**
   * *Descriere:* Comentarii deschise, neutre sau pro-occidentale care susțin cooperarea internațională și parteneriatele strategice (NATO, UE), fără a folosi identitatea națională ca barieră.
   * *Exemplu din text:* Comentariul în care sunt lăudate analizele clarvăzătoare ale domnului Corneliu Bjola.

2. **Suveranistul Geopolitic (Scor 1-2 pe Geopolitic, Scor 0 pe Identitate)**
   * *Descriere:* Discurs centrat strict pe nemulțumiri de natură economică, politică sau militară externă. Manifestă izolaționism pur geopolitic (critici aduse SUA, NATO sau ajutorului pentru Ucraina), dar fără a apela la o retorică culturală sau etnică explicită.
   * *Exemplu din text:* Comentariul care acuză SUA că „a mâncat banii țărilor NATO și le-a obligat să cumpere armament”.

3. **Naționalistul Cultural (Scor 0 pe Geopolitic, Scor 1-2 pe Identitate)**
   * *Descriere:* Focusat pe protejarea identității, a tradițiilor sau a purității naționale în fața globalizării, simțindu-se amenințat de multiculturalism, dar fără a formula o agendă sau o critică geopolitică directă la adresa alianțelor externe în acel comentariu.

4. **Suveranistul Radical / Ultra-naționalistul (Scor 2 pe ambele axe)**
   * *Descriere:* Intersecția maximă dintre izolaționismul geopolitic radical și etnocentrismul rigid. Respingerea totală și agresivă a Occidentului și a structurilor supranaționale este combinată direct cu argumente de superioritate sau victimizare absolută a identității naționale în fața „străinilor”.